# 10: ISIC 2019 Bronze ingestion

Indexes every image in the archives (checksum and location, no bytes) and loads the metadata as published. Run after `05_setup_tables_and_folders`, with the archives uploaded.

Produces `bronze.isic_2019_image_index`, `bronze.isic_2019_source_metadata` and a row in `bronze.ingestion_runs`.

In [ ]:
%run ../_setup_env


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

from pyspark.sql import functions as F

from data_platform.files import check_archives_exist, stage_archives_and_extract_metadata
from data_platform.dataset_layout import build_layout, load_dataset_config, load_yaml_config, resolve_archive_paths
from data_platform.spark_io import (
    load_combined_metadata,
    print_bronze_outputs,
    write_image_index_table,
    write_ingestion_run,
)

import tempfile


DATASET_CONFIG = load_dataset_config(CONFIG_ROOT / "bronze" / "datasets" / "isic_2019.yaml")
STORAGE_CONFIG = load_yaml_config(CONFIG_ROOT / "storage.yaml")
DATASET_CONFIG["landing_root"] = STORAGE_CONFIG["landing_root"]
DATASET_CONFIG["storage_root"] = STORAGE_CONFIG["storage_root"]
layout = build_layout(DATASET_CONFIG)
landing_paths = layout["landing_paths"]
bronze_paths = layout["bronze_paths"]
bronze_tables = layout["bronze_tables"]

ARCHIVES = resolve_archive_paths(DATASET_CONFIG["archives"], landing_paths, bronze_paths)
LOCAL_ARCHIVE_STAGE_ROOT = Path(tempfile.gettempdir()) / DATASET_CONFIG["dataset_key"] / "archive_stage"
OVERWRITE_LOCAL_STAGE = False

for archive in ARCHIVES:
    print(f"{archive['source_split']} archive landing path: {archive['archive_dbfs_path']}")
    print(f"{archive['source_split']} Bronze image index table: {bronze_tables['image_index']}")
    print(f"{archive['source_split']} Bronze metadata path: {archive['metadata_target_path']}")


## Check the archives are in landing

In [ ]:
check_archives_exist(ARCHIVES)


## Stage archives locally and extract the metadata CSV

Each archive is copied once to local disk, so no step makes one cloud request per image.

In [ ]:
stage_archives_and_extract_metadata(ARCHIVES, LOCAL_ARCHIVE_STAGE_ROOT, OVERWRITE_LOCAL_STAGE)

display(dbutils.fs.ls(bronze_paths["metadata"]))


## Write the image index

Checksums and archive locations only; image bytes are read and discarded ([ADR 004](../../docs/decisions/004-stream-archives-no-blob-storage.md)).

In [ ]:
image_manifest_df = write_image_index_table(spark, ARCHIVES, bronze_tables["image_index"])


## Write the source metadata and log the run

The metadata schema below is ISIC 2019's own; it's the one dataset-specific step in Bronze.

In [ ]:
raw_metadata_df = load_combined_metadata(spark, ARCHIVES)

ingestion_run_id = f"bronze-{uuid4().hex[:12]}"
started_at = datetime.now(timezone.utc)

# ISIC 2019's actual Bronze source metadata schema — dataset-specific by design, unlike
# everything else in this notebook (see docs/data_contract.md).
source_metadata_df = (
    raw_metadata_df
    .withColumn("image_id", F.col("isic_id").cast("string"))
    .join(image_manifest_df, on=["image_id", "source_split"], how="left")
    .select(
        F.col("image_id"),
        F.col("source_split"),
        F.col("source_uri"),
        F.col("attribution"),
        F.col("copyright_license"),
        F.col("age_approx"),
        F.col("anatom_site_1"),
        F.col("anatom_site_2"),
        F.col("anatom_site_3"),
        F.col("anatom_site_4"),
        F.col("anatom_site_5"),
        F.col("anatom_site_special"),
        F.col("clin_size_long_diam_mm"),
        F.col("concomitant_biopsy"),
        F.col("dermoscopic_type"),
        F.col("diagnosis_1"),
        F.col("diagnosis_2"),
        F.col("diagnosis_3"),
        F.col("diagnosis_4"),
        F.col("diagnosis_5"),
        F.col("diagnosis_confirm_type"),
        F.col("family_hx_mm"),
        F.col("image_type"),
        F.col("lesion_id"),
        F.col("melanocytic"),
        F.col("patient_id"),
        F.col("personal_hx_mm"),
        F.col("sex"),
        F.col("source_checksum"),
    )
    .where(F.col("image_id").isNotNull())
    .withColumn("ingestion_run_id", F.lit(ingestion_run_id))
    .withColumn("ingested_at", F.lit(started_at))
)

source_metadata_df.createOrReplaceTempView("staged_isic_2019_source_metadata")
spark.sql(
    f"""
    MERGE INTO {bronze_tables['source_metadata']} AS target
    USING staged_isic_2019_source_metadata AS source
    ON target.image_id = source.image_id AND target.source_split = source.source_split
    WHEN MATCHED THEN UPDATE SET
      target.source_split = source.source_split,
      target.source_uri = source.source_uri,
      target.attribution = source.attribution,
      target.copyright_license = source.copyright_license,
      target.age_approx = source.age_approx,
      target.anatom_site_1 = source.anatom_site_1,
      target.anatom_site_2 = source.anatom_site_2,
      target.anatom_site_3 = source.anatom_site_3,
      target.anatom_site_4 = source.anatom_site_4,
      target.anatom_site_5 = source.anatom_site_5,
      target.anatom_site_special = source.anatom_site_special,
      target.clin_size_long_diam_mm = source.clin_size_long_diam_mm,
      target.concomitant_biopsy = source.concomitant_biopsy,
      target.dermoscopic_type = source.dermoscopic_type,
      target.diagnosis_1 = source.diagnosis_1,
      target.diagnosis_2 = source.diagnosis_2,
      target.diagnosis_3 = source.diagnosis_3,
      target.diagnosis_4 = source.diagnosis_4,
      target.diagnosis_5 = source.diagnosis_5,
      target.diagnosis_confirm_type = source.diagnosis_confirm_type,
      target.family_hx_mm = source.family_hx_mm,
      target.image_type = source.image_type,
      target.lesion_id = source.lesion_id,
      target.melanocytic = source.melanocytic,
      target.patient_id = source.patient_id,
      target.personal_hx_mm = source.personal_hx_mm,
      target.sex = source.sex,
      target.source_checksum = source.source_checksum,
      target.ingestion_run_id = source.ingestion_run_id,
      target.ingested_at = source.ingested_at
    WHEN NOT MATCHED THEN INSERT *
    """
)

records_written = source_metadata_df.count()
finished_at = datetime.now(timezone.utc)
write_ingestion_run(
    spark,
    bronze_tables,
    ingestion_run_id,
    DATASET_CONFIG["dataset_name"],
    DATASET_CONFIG["source_version"],
    started_at,
    finished_at,
    records_seen=records_written,
    records_written=records_written,
)

print(f"Ingestion run id: {ingestion_run_id}")
print(f"Records written: {records_written}")


## Review

In [ ]:
print_bronze_outputs(spark, dbutils, display, landing_paths, bronze_paths, bronze_tables, DATASET_CONFIG["dataset_name"])
